___
# <center>Condicional, independência e Bayes</center>
___

## Aula 08

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * montar a tabela de dupla entrada e ler condicionais nela;
 * calcular uma condicional filtrando e recontando;
 * testar independência comparando o observado com o esperado;
 * inverter uma condicional com o teorema de Bayes.

Curto, como sempre. Tudo que está aqui já foi feito na lousa hoje.

Uma diferença em relação ao slide: lá a tabela dos 300 processos era
**ilustrativa**, com números escolhidos para fechar de cabeça. Aqui a base é
**real**, e por isso as divisões não dão números redondos. É assim que a conta
aparece na vida.


___
<div id="indice"></div>

## Índice

- [A base de hoje](#dados)

- [A tabela de dupla entrada](#tabela)

- [Probabilidade condicional](#condicional)

- [Independência: o teste](#independencia)

- [Teorema de Bayes](#bayes)

- [Variável aleatória e esperança](#esperanca)

- [RESUMO](#resumo)


___
<div id="dados"></div>

# A base de hoje

Acórdãos criminais do TJSP. Uma linha por acórdão.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"

criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")

# Fora as linhas sem regime: não dá para calcular proporção de regime
# em acórdão que não informou regime nenhum.
penas = criminal.dropna(subset=["regime_inicial"])

penas.shape


Duas informações interessam hoje:

- `houve_reincidencia`: o acórdão registrou reincidência;
- `regime_inicial`: fechado, semiaberto ou aberto.

O regime tem três valores, e a aula toda foi sobre eventos de sim ou não. Então
a primeira coisa é transformar "regime" no evento **"o regime é fechado"**.


In [ ]:
penas = penas.assign(fechado=penas["regime_inicial"] == "fechado")

penas[["houve_reincidencia", "regime_inicial", "fechado"]].head()


[Volta ao Índice](#indice)


___
<div id="tabela"></div>

# A tabela de dupla entrada

`pd.crosstab` cruza duas colunas e conta quantos casos caem em cada combinação.
É a mesma tabela do slide, agora com os números da base.

- o **miolo** é a distribuição **conjunta**;
- a última linha e a última coluna são as **marginais**.


In [ ]:
pd.crosstab(
    penas["houve_reincidencia"],
    penas["fechado"],
    margins=True,
    margins_name="total",
)


✔️ **A mesma tabela em proporções.** `normalize="all"` divide tudo pelo total
geral, e é a versão em que o canto vale 1.


In [ ]:
pd.crosstab(
    penas["houve_reincidencia"],
    penas["fechado"],
    margins=True,
    margins_name="total",
    normalize="all",
).round(3)


**✍️ Agora você.** Nessa tabela de proporções, aponte:

1. a **conjunta** de "reincidente e regime fechado";
2. a **marginal** de "regime fechado".


[Volta ao Índice](#indice)


___
<div id="condicional"></div>

# Probabilidade condicional

Condicionar é trocar o denominador: em vez de dividir pelo total, dividimos só
pelo grupo que interessa. No pandas isso é **filtrar e recontar**.

Comece pela probabilidade sem condicionar. A média de uma coluna booleana **é**
a proporção de `True`, porque `True` vale 1 e `False` vale 0.


In [ ]:
# P(fechado), sem condicionar
penas["fechado"].mean()


In [ ]:
# P(fechado | reincidente): filtra, e recalcula a média dentro do filtro
reincidentes = penas.query("houve_reincidencia")

reincidentes["fechado"].mean()


**✍️ Agora você.** O outro lado:
$P(\text{fechado} \mid \text{não reincidente})$.


In [ ]:
nao_reincidentes = penas.query("~houve_reincidencia")

nao_reincidentes["________"].mean()


Três números: cerca de 0,46 sem condicionar, 0,61 entre reincidentes e 0,35
entre não reincidentes. **Saber sobre a reincidência muda a probabilidade**, e é
isso que significa dizer que as duas variáveis têm relação.

⚠️ **Cuidado com o `normalize`.** `"index"` divide por linha, `"columns"` por
coluna e `"all"` pelo total. As três dão números diferentes e respondem a
perguntas diferentes. Errar aqui é o mesmo erro de dividir pelo total em vez de
pelo grupo.


In [ ]:
# todas as condicionais de uma vez: cada linha soma 1
pd.crosstab(
    penas["houve_reincidencia"],
    penas["fechado"],
    normalize="index",
).round(3)


[Volta ao Índice](#indice)


___
<div id="independencia"></div>

# Independência: o teste

Se dois eventos fossem independentes, a probabilidade dos dois juntos seria o
produto das marginais. O teste é comparar esse produto com o que a base tem.


In [ ]:
esperado = penas["fechado"].mean() * penas["houve_reincidencia"].mean() * len(penas)
observado = (penas["fechado"] & penas["houve_reincidencia"]).sum()

print("esperado sob independência:", round(esperado, 1))
print("observado                 :", observado)


Cerca de 62 contra 83. O observado é bem maior que o esperado, então os eventos
**não** são independentes.

💡 E, como na aula: isso é **associação**, não causa. Reincidência e regime
fechado andam juntos, e a lei explica boa parte disso. O número sozinho não diz
qual é a explicação.


[Volta ao Índice](#indice)


___
<div id="bayes"></div>

# Teorema de Bayes

Bayes inverte a condicional:

$$P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}$$

Com a base inteira na mão, dá para conferir que ele bate com a conta direta.


In [ ]:
p_reincidencia = penas["houve_reincidencia"].mean()
p_fechado = penas["fechado"].mean()
p_fechado_dado_reincidencia = reincidentes["fechado"].mean()

# Bayes: P(reincidente | fechado)
bayes = p_fechado_dado_reincidencia * p_reincidencia / p_fechado

# a conta direta, filtrando
direto = penas.query("fechado")["houve_reincidencia"].mean()

print("por Bayes :", round(bayes, 4))
print("direto    :", round(direto, 4))


Os dois dão o mesmo número, e é assim que tem que ser: Bayes não é uma conta
nova, é a regra do produto escrita de outro jeito.

Repare que $P(\text{fechado} \mid \text{reincidente}) \approx 0,61$ e
$P(\text{reincidente} \mid \text{fechado}) \approx 0,55$ são **números
diferentes**. Trocar um pelo outro é o erro do promotor, e aqui a troca custaria
seis pontos percentuais.

Bayes importa quando você **não tem a base inteira** para filtrar, e só conhece
$P(B \mid A)$ e as marginais. É a situação da perícia: o laudo informa a taxa
de erro do exame, e ninguém tem a tabela do lote inteiro.


<div id="ex1"></div>

### EXERCÍCIO 1

A perícia grafotécnica da aula, agora em código.

Dois eventos, e só eles:

- $F$: a assinatura do contrato **é falsa**;
- $A$: a perícia **aponta** falsidade nesse contrato.

O enunciado da lousa dá três números: $P(F) = 0{,}01$,
$P(A \mid F) = 0{,}99$ e $P(A \mid F^c) = 0{,}01$. E pede $P(F \mid A)$.

1. calcule $P(A)$ pela lei da probabilidade total;
2. calcule $P(F \mid A)$ por Bayes, e confira com os 50% da lousa;
3. refaça com $P(F) = 0{,}50$, como se a perícia só fosse pedida em contratos já
   sob suspeita. O que acontece com a resposta?


In [ ]:
def p_falso_dado_apontado(p_falso,
                          p_aponta_dado_falso=0.99,
                          p_aponta_dado_autentico=0.01):
    # lei da probabilidade total: os apontados saem dos falsos e dos autênticos
    p_aponta = (p_aponta_dado_falso * p_falso
                + ________ * (1 - p_falso))
    # Bayes
    return p_aponta_dado_falso * p_falso / ________

for antes in (0.01, 0.10, 0.50):
    print(f"P(F) = {antes:.0%}  ->  P(F | A) = {p_falso_dado_apontado(antes):.1%}")


💡 A perícia é a mesma nas três linhas, e o laudo diria exatamente a mesma
coisa. O que muda é **em que lote ela foi aplicada**.


[Volta ao Índice](#indice)


___
<div id="esperanca"></div>

# Variável aleatória e esperança

Uma **variável aleatória** é uma função que leva elementos do espaço amostral em
valores numéricos, e para cada valor sabemos atribuir uma probabilidade. Essa
lista de valores com as probabilidades deles é a **distribuição**.

No caso dos honorários de êxito, visto no fim da aula: duas audiências por dia,
20% de chance de acordo em cada uma, e R$ 500 por acordo fechado.

| $x$ | $P(X = x)$ |
|---|---|
| 0 | 0,64 |
| 500 | 0,32 |
| 1000 | 0,04 |

A **esperança** é a média dos valores possíveis, cada um pesado pela sua
probabilidade.


In [ ]:
import numpy as np

valores = np.array([0, 500, 1000])          # honorários do dia, em reais
probabilidades = np.array([0.64, 0.32, 0.04])

esperanca = (valores * probabilidades).sum()

print("as probabilidades somam:", probabilidades.sum())
print("honorário esperado por dia:", esperanca, "reais")


200 reais **não é um resultado possível**: num dia ela ganha 0, 500 ou 1000. A
esperança é o que sai na média ao longo de muitos dias.

E a Bernoulli fecha o círculo: quando $X$ vale 1 ou 0, a esperança é $p$. Por
isso a média de uma coluna booleana, que usamos a aula inteira, **é** uma
esperança.


In [ ]:
# a media de uma coluna booleana E a esperanca de uma Bernoulli
penas["fechado"].mean()


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

| ideia | no pandas |
|---|---|
| $P(A)$ | média de uma coluna booleana |
| tabela de dupla entrada | `pd.crosstab(a, b, margins=True)` |
| a mesma tabela em proporções | `normalize="all"` |
| $P(A \mid B)$ | `.query()` no B, e a média de A dentro do filtro |
| todas as condicionais de uma vez | `normalize="index"` ou `"columns"` |
| independência | comparar $P(A)P(B)n$ com o observado |
| Bayes | $P(B \mid A)P(A)/P(B)$, com $P(B)$ pela marginal |
| esperança | soma de valor vezes probabilidade |

**A frase para levar:** condicionar é trocar o denominador, e Bayes é o que
permite trocar de volta.


[Volta ao Índice](#indice)
